<div dir="rtl" style="text-align: right;">

# آموزش الگوریتم‌های یادگیری ماشین: 

**توضیح پروژه**: این پروژه به آموزش الگوریتم‌های یادگیری ماشین شامل **درخت تصمیم (Decision Tree)**، **ماشین بردار پشتیبان (SVM)**، **K-نزدیک‌ترین همسایه (KNN)**، و **مدل‌های مبتنی بر درخت (Random Forest و Gradient Boosting)** می‌پردازد. از دیتاست **Iris** برای سادگی و آشنایی با این الگوریتم‌ها استفاده می‌کنیم. دیتاست Iris شامل 150 نمونه گل با 4 ویژگی (طول و عرض کاسبرگ و گلبرگ) و 3 کلاس (نوع گل) است.

**اهداف**: درک مفاهیم الگوریتم‌های یادگیری ماشین، پیاده‌سازی آن‌ها با Scikit-Learn، و مقایسه عملکردشان.

**دستورالعمل دانشجویان**: کدها را اجرا کنید، نتایج را تحلیل کنید، و عملکرد مدل‌ها را مقایسه کنید.
<div/>

<div dir="rtl" style="text-align: right;">

## مرحله ۱: بارگذاری و بررسی داده‌ها
دیتاست Iris را از Scikit-Learn بارگذاری کرده و بررسی اولیه انجام می‌دهیم.
<div/>

In [16]:
from sklearn.datasets import load_iris
import pandas as pd

# بارگذاری دیتاست
iris = load_iris()
data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
data['target'] = iris.target

# بررسی اولیه
print("5 ردیف اول:\n", data.head())
print("\nاطلاعات دیتاست:\n")
data.info()
print("\nآمار توصیفی:\n", data.describe())

5 ردیف اول:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

اطلاعات دیتاست:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 no

<div dir="rtl" style="text-align: right;">

**تفسیر**: دیتاست شامل 150 نمونه با 4 ویژگی عددی و 3 کلاس (0, 1, 2) است. توزیع کلاس‌ها متعادل است (50 نمونه برای هر کلاس).
<div/>

<div dir="rtl" style="text-align: right;">

## مرحله ۲: پیش‌پردازش داده‌ها
ویژگی‌ها را استانداردسازی می‌کنیم.
<div/>

In [17]:
from sklearn.preprocessing import StandardScaler

# ویژگی‌ها و برچسب
X = data.drop('target', axis=1)
y = data['target']

# استانداردسازی
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("5 نمونه اول استانداردشده:\n", X_scaled[:5])

5 نمونه اول استانداردشده:
 [[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]
 [-1.50652052  0.09821729 -1.2833891  -1.3154443 ]
 [-1.02184904  1.24920112 -1.34022653 -1.3154443 ]]


<div dir="rtl" style="text-align: right;">

**تفسیر**: استانداردسازی باعث شد ویژگی‌ها میانگین 0 و انحراف معیار 1 داشته باشند، که برای الگوریتم‌هایی مثل SVM و KNN مهم است.
<div/>

<div dir="rtl" style="text-align: right;">

## مرحله ۳: تقسیم داده‌ها به آموزشی، اعتبارسنجی، و تست
داده‌ها را به 70% آموزشی، 15% اعتبارسنجی، و 15% تست تقسیم می‌کنیم.
<div/>

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
print("اندازه آموزشی:", X_train.shape)
print("اندازه تست:", X_test.shape)

اندازه آموزشی: (105, 4)
اندازه تست: (45, 4)


<div dir="rtl" style="text-align: right;">

**تفسیر**: داده‌ها با حفظ توزیع کلاس‌ها (به دلیل `stratify`) تقسیم شدند تا از سوگیری جلوگیری شود.
<div/>

<div dir="rtl" style="text-align: right;">

## مرحله ۴: الگوریتم‌ها
الگوریتم‌های درخت تصمیم، SVM، KNN، Random Forest، و Gradient Boosting را پیاده‌سازی و مقایسه می‌کنیم.
<div/>

<div dir="rtl" style="text-align: right;">

### ۴.۱: درخت تصمیم (Decision Tree)

**ویژگی‌های مدل**:
- درخت تصمیم یک مدل طبقه‌بندی یا رگرسیون است که داده‌ها را با سوالات باینری (مثل "آیا سن > 30؟") به شاخه‌ها تقسیم می‌کند تا بهترین جداسازی کلاس‌ها را پیدا کند.
- **مزایا**: ساده و قابل تفسیر (می‌توان درخت را به صورت گرافیکی دید)، نیاز به پیش‌پردازش کم، و مدیریت داده‌های غیرخطی.
- **معایب**: ممکن است بیش‌برازش کند (درخت خیلی عمیق شود)، حساس به تغییرات کوچک داده‌ها، و عملکرد ضعیف در داده‌های نامتعادل.

**پارامترهای مشخص‌شده**:
- `random_state=42`: برای تکرارپذیری نتایج (کنترل تصادفی بودن).
- می‌توان پارامترهایی مثل `max_depth` (حداکثر عمق درخت برای جلوگیری از بیش‌برازش) را قرار داد.

max_depth: حداکثر عمق درخت تصمیم را مشخص می‌کند.
هرچه عمق بیشتر باشد، مدل پیچیده‌تر و دقیق‌تر روی داده‌های آموزش می‌شود (ولی ممکن است overfitting کند). عمق کم یعنی مدل ساده‌تر و احتمالاً خطای کمتر روی داده‌های جدید.

مدل درخت تصمیم را آموزش داده و ارزیابی می‌کنیم.
<div/>

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# مدل درخت تصمیم
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# ارزیابی روی تست
y_test_pred_dt = dt_model.predict(X_test)
print("دقت درخت تصمیم (تست):", accuracy_score(y_test, y_test_pred_dt))
print("\nگزارش طبقه‌بندی (تست):\n", classification_report(y_test, y_test_pred_dt))

دقت درخت تصمیم (تست): 0.9333333333333333

گزارش طبقه‌بندی (تست):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      0.80      0.89        15
           2       0.83      1.00      0.91        15

    accuracy                           0.93        45
   macro avg       0.94      0.93      0.93        45
weighted avg       0.94      0.93      0.93        45



<div dir="rtl" style="text-align: right;">

**تفسیر**: درخت تصمیم با تقسیم‌بندی داده‌ها بر اساس ویژگی‌ها عمل می‌کند. ممکن است به دلیل سادگی بیش‌برازش کند، اما برای دیتاست Iris که داده‌ها به‌خوبی قابل تفکیک هستند، عملکرد خوبی دارد.

تفاوت ارزیابی روی اعتبارسنجی و تست:

مجموعه اعتبارسنجی (Validation Set): برای تنظیم مدل (مثل انتخاب بهترین پارامترها یا مدل) استفاده می‌شود. مدل روی داده‌های آموزشی یاد می‌گیرد و روی اعتبارسنجی ارزیابی می‌شود تا عملکردش بهبود یابد. این مجموعه به ما کمک می‌کند تا مدل را بدون دیدن داده‌های تست بهینه کنیم.

مجموعه تست (Test Set): برای ارزیابی نهایی عملکرد مدل استفاده می‌شود. این مجموعه فقط یک بار پس از نهایی شدن مدل استفاده می‌شود تا عملکرد واقعی مدل روی داده‌های جدید و نادیده بررسی شود.

خلاصه: اعتبارسنجی برای تنظیم و بهبود مدل در طول فرآیند آموزش است، اما تست برای سنجش نهایی عملکرد مدل روی داده‌های کاملاً مستقل استفاده می‌شود.
<div/>

<div dir="rtl" style="text-align: right;">

### ۴.۲: ماشین بردار پشتیبان (SVM)

**ویژگی‌های مدل**:
- SVM با یافتن بهترین مرز جداسازی (hyperplane) کلاس‌ها را جدا می‌کند و بیشترین حاشیه را بین کلاس‌ها ایجاد می‌کند.
- **مزایا**: در فضای با ابعاد بالا خوب عمل می‌کند، با کرنل‌های مختلف (خطی، RBF) برای داده‌های غیرخطی مناسب است، و به پرت‌ها مقاوم است.
- **معایب**: برای داده‌های بزرگ کند است، نیاز به تنظیم کرنل و پارامتر C (برای کنترل حاشیه و خطا) دارد، و تفسیر سختی دارد.

**پارامترهای مشخص‌شده**:
- `kernel='linear'`: کرنل خطی برای جداسازی خطی کلاس‌ها.
- `random_state=42`: برای تکرارپذیری.
- `probability=True`: برای محاسبه احتمال کلاس‌ها.
- می‌توان پارامترهایی مثل `C` (برای کنترل خطاها) یا `kernel='rbf'` (برای داده‌های غیرخطی) را تنظیم کرد.

مدل SVM را با کرنل خطی آموزش داده و ارزیابی می‌کنیم.
<div/>

In [20]:
from sklearn.svm import SVC

# مدل SVM
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_model.fit(X_train, y_train)

# ارزیابی روی تست
y_test_pred_svm = svm_model.predict(X_test)
print("دقت SVM (تست):", accuracy_score(y_test, y_test_pred_svm))
print("\nگزارش طبقه‌بندی (تست):\n", classification_report(y_test, y_test_pred_svm))

دقت SVM (تست): 0.9111111111111111

گزارش طبقه‌بندی (تست):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.82      0.93      0.88        15
           2       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



<div dir="rtl" style="text-align: right;">

**تفسیر**: SVM با یافتن بهترین مرز جداسازی، برای دیتاست Iris که کلاس‌ها نسبتاً قابل تفکیک هستند، عملکرد قوی دارد. کرنل خطی برای این داده‌های ساده کافی است.

Kernel: یک تابع ریاضی است که داده‌ها را به فضای جدیدی منتقل می‌کند تا راحت‌تر بتوان بین کلاس‌ها مرز تصمیم کشید. مثلاً اگر داده‌ها در حالت عادی خطی جدا نشوند، با kernel می‌توان آن‌ها را در فضای بالاتری خطی جدا کرد. (مثل RBF، پلی‌نومیال، خطی).

Probability: SVM به طور پیش‌فرض فقط می‌گوید "این داده به کدام کلاس تعلق دارد"، ولی با فعال کردن probability می‌توان تخمینی از احتمال تعلق داده به هر کلاس گرفت (مثلاً 70% کلاس A، 30% کلاس B).
<div/>

<div dir="rtl" style="text-align: right;">

### ۴.۳: K-نزدیک‌ترین همسایه (KNN)

**ویژگی‌های مدل**:
- KNN بر اساس نزدیک‌ترین همسایگان (k نمونه نزدیک) کلاس یک نمونه جدید را پیش‌بینی می‌کند. از فاصله (مثل اقلیدسی) استفاده می‌کند.
- **مزایا**: ساده، بدون فرض توزیع داده‌ها، و برای داده‌های کوچک خوب عمل می‌کند.
- **معایب**: برای داده‌های بزرگ کند است (نیاز به محاسبه فاصله برای همه نمونه‌ها)، حساس به نویز و مقیاس داده‌ها، و نیاز به انتخاب k مناسب دارد.

**پارامترهای مشخص‌شده**:
- `n_neighbors=5`: تعداد همسایگان (k=5) برای پیش‌بینی.
- n_neighbors: تعیین می‌کند چند همسایه‌ی نزدیک برای تصمیم‌گیری در نظر گرفته شوند.
مثلا اگر n_neighbors=5 باشد، الگوریتم به 5 همسایه نزدیک نگاه می‌کند و بر اساس رأی‌گیری یا میانگین، کلاس یا مقدار خروجی را مشخص می‌کند.

مدل KNN را با k=5 آموزش داده و ارزیابی می‌کنیم.
<div/>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# مدل KNN
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)

# ارزیابی روی تست
y_test_pred_knn = knn_model.predict(X_test)
print("دقت KNN (تست):", accuracy_score(y_test, y_test_pred_knn))
print("\nگزارش طبقه‌بندی (تست):\n", classification_report(y_test, y_test_pred_knn))

دقت KNN (تست): 0.9111111111111111

گزارش طبقه‌بندی (تست):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.79      1.00      0.88        15
           2       1.00      0.73      0.85        15

    accuracy                           0.91        45
   macro avg       0.93      0.91      0.91        45
weighted avg       0.93      0.91      0.91        45



<div dir="rtl" style="text-align: right;">

**تفسیر**: KNN با استفاده از فاصله‌های نمونه‌ها، برای دیتاست Iris که داده‌ها به‌خوبی قابل تفکیک هستند، معمولاً دقت بالایی دارد. انتخاب k=5 تعادل خوبی بین دقت و تعمیم‌پذیری ایجاد می‌کند.
<div/>

<div dir="rtl" style="text-align: right;">

### ۴.۴: Random Forest

**ویژگی‌های مدل**:
- Random Forest مجموعه‌ای از درخت‌های تصمیم است که با ترکیب پیش‌بینی‌های چندین درخت (bagging) عمل می‌کند و تنوع را با انتخاب تصادفی ویژگی‌ها افزایش می‌دهد.
- **مزایا**: دقت بالا، کاهش بیش‌برازش نسبت به درخت تصمیم تکی، مدیریت داده‌های نامتعادل، و شناسایی اهمیت ویژگی‌ها.
- **معایب**: پیچیده‌تر و کندتر از درخت تصمیم، تفسیر سختی دارد (جنگل درخت‌ها).

**پارامترهای مشخص‌شده**:
- `n_estimators=100`: تعداد درخت‌ها (100 درخت).
n_estimators: تعداد درخت‌هایی که باید ساخته شوند. هرچه تعداد بیشتر باشد، مدل پایدارتر و دقیق‌تر می‌شود (ولی محاسبات بیشتر لازم دارد). در واقع خروجی نهایی ترکیب پیش‌بینی همه‌ی این درخت‌هاست.

- `random_state=42`: برای تکرارپذیری.
- می‌توان پارامترهایی مثل `max_depth` (عمق درخت‌ها)  را تنظیم کرد.

مدل Random Forest را آموزش می‌دهیم.
<div/>

In [22]:
from sklearn.ensemble import RandomForestClassifier

# مدل Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# ارزیابی روی تست
y_test_pred_rf = rf_model.predict(X_test)
print("دقت Random Forest (تست):", accuracy_score(y_test, y_test_pred_rf))
print("\nگزارش طبقه‌بندی (تست):\n", classification_report(y_test, y_test_pred_rf))

دقت Random Forest (تست): 0.8888888888888888

گزارش طبقه‌بندی (تست):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.78      0.93      0.85        15
           2       0.92      0.73      0.81        15

    accuracy                           0.89        45
   macro avg       0.90      0.89      0.89        45
weighted avg       0.90      0.89      0.89        45



<div dir="rtl" style="text-align: right;">

**تفسیر**: Random Forest با ترکیب چندین درخت، معمولاً دقت و پایداری بیشتری نسبت به درخت تصمیم تکی دارد. برای دیتاست Iris، عملکرد نزدیک به کامل است.
<div/>

<div dir="rtl" style="text-align: right;">

### ۴.۵: Gradient Boosting

**ویژگی‌های مدل**:
- Gradient Boosting مجموعه‌ای از درخت‌های تصمیم است که به‌صورت متوالی ساخته می‌شود و هر درخت خطاهای درخت قبلی را تصحیح می‌کند (boosting).
- **مزایا**: دقت بالا، مدیریت داده‌های پیچیده و غیرخطی، و عملکرد خوب در مسائل نامتعادل.
- **معایب**: ممکن است بیش‌برازش کند اگر پارامترها تنظیم نشوند، کندتر از Random Forest، و نیاز به تنظیم دقیق دارد.

**پارامترهای مشخص‌شده**:
- `n_estimators=100`: تعداد مراحل boosting (100 درخت).
- `random_state=42`: برای تکرارپذیری.
- می‌توان پارامترهایی مثل `learning_rate` (سرعت یادگیری) یا `max_depth` (عمق درخت‌ها) را تنظیم کرد.

مدل Gradient Boosting را آموزش می‌دهیم.

برخلاف Random Forest (که درخت‌ها را مستقل و موازی می‌سازد)، Gradient Boosting درخت‌ها را به صورت ترتیبی می‌سازد.
هر درخت جدید برای "اصلاح خطاهای درخت قبلی" ساخته می‌شود. به این ترتیب مدل به تدریج بهتر می‌شود.

n_estimators: تعداد درخت‌هایی که پشت سر هم ساخته می‌شوند.
(اگر خیلی زیاد باشد ممکن است overfitting رخ دهد، اگر خیلی کم باشد مدل کافی یاد نمی‌گیرد).

learning_rate: میزان تأثیر هر درخت جدید روی مدل نهایی.
اگر بزرگ باشد، هر درخت جدید تغییر زیادی در مدل می‌دهد (یادگیری سریع‌تر ولی خطر overfitting).
اگر کوچک باشد، یادگیری کندتر است ولی معمولاً مدل دقیق‌تر و پایدارتر می‌شود.
<div/>

In [23]:
from sklearn.ensemble import GradientBoostingClassifier

# مدل Gradient Boosting
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)

# ارزیابی روی تست
y_test_pred_gb = gb_model.predict(X_test)
print("دقت Gradient Boosting (تست):", accuracy_score(y_test, y_test_pred_gb))
print("\nگزارش طبقه‌بندی (تست):\n", classification_report(y_test, y_test_pred_gb))

دقت Gradient Boosting (تست): 0.9333333333333333

گزارش طبقه‌بندی (تست):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       0.88      0.93      0.90        15
           2       0.93      0.87      0.90        15

    accuracy                           0.93        45
   macro avg       0.93      0.93      0.93        45
weighted avg       0.93      0.93      0.93        45



<div dir="rtl" style="text-align: right;">
**تفسیر**: Gradient Boosting با یادگیری تدریجی و تمرکز بر خطاها، معمولاً دقت بالایی برای دیتاست‌هایی مثل Iris دارد. برای جلوگیری از بیش‌برازش، می‌توان `learning_rate` را کاهش داد.
<div/>

<div dir="rtl" style="text-align: right;">
## مرحله ۵: مقایسه مدل‌ها
عملکرد مدل‌ها را مقایسه می‌کنیم.
<div/>

In [24]:
# مقایسه دقت مدل‌ها
models = {
    'Decision Tree': accuracy_score(y_test, y_test_pred_dt),
    'SVM': accuracy_score(y_test, y_test_pred_svm),
    'KNN': accuracy_score(y_test, y_test_pred_knn),
    'Random Forest': accuracy_score(y_test, y_test_pred_rf),
    'Gradient Boosting': accuracy_score(y_test, y_test_pred_gb)
}

print("مقایسه دقت مدل‌ها روی مجموعه تست:\n", pd.Series(models))

مقایسه دقت مدل‌ها روی مجموعه تست:
 Decision Tree        0.933333
SVM                  0.911111
KNN                  0.911111
Random Forest        0.888889
Gradient Boosting    0.933333
dtype: float64
